# Chunk Strategies in RAG

In [1]:
# !uv add -r requirements.txt
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_core.documents import Document

text = """
The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference. 
Early systems focused on symbolic reasoning and rule-based approaches. However, progress slowed during the AI winters in the 1970s and 1980s due to limited computing power and data availability.

A major breakthrough occurred in the 2010s with the rise of deep learning and large-scale datasets. Companies like OpenAI, Google, and Meta developed large language models such as GPT series. 
These models can now generate human-like text, write code, create images, and perform complex reasoning.

Despite these advances, significant challenges remain including ethical concerns, bias in training data, high energy consumption, and the urgent need for robust regulatory frameworks.
"""

docs = [Document(page_content=text)]
print(f"Document loaded ({len(text)} characters)")

Document loaded (833 characters)


## Fixed-Width Chunking

In [4]:
def fixed_width_chunking(text, chunk_size=200, overlap=20):
    """Splits text into fixed-size chunks with optional overlap."""
    
    chunks = [] # List to hold the resulting chunks
    start = 0 # Starting index for chunking
    while start < len(text):
        # Calculate the end index for the current chunk
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)
        start += (chunk_size - overlap)
    print(f"Created {len(chunks)} chunks using **Fixed-Width Chunking**")
    return chunks

# Execute
fixed_chunks = fixed_width_chunking(text)
for i, chunk in enumerate(fixed_chunks):
    print(f"[cyan] Fixed Chunk {i+1}: [/cyan] {chunk}..." if len(chunk) > 180 else chunk)

Created 5 chunks using **Fixed-Width Chunking**
[cyan] Fixed Chunk 1: [/cyan] 
The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference. 
Early systems focused on symbolic reasoning an...
[cyan] Fixed Chunk 2: [/cyan] ymbolic reasoning and rule-based approaches. However, progress slowed during the AI winters in the 1970s and 1980s due to limited computing power and data availability.

A major breakthrough occurred ...
[cyan] Fixed Chunk 3: [/cyan] eakthrough occurred in the 2010s with the rise of deep learning and large-scale datasets. Companies like OpenAI, Google, and Meta developed large language models such as GPT series. 
These models can ...
[cyan] Fixed Chunk 4: [/cyan] . 
These models can now generate human-like text, write code, create images, and perform complex reasoning.

Despite these advances, significant challenges remain including ethical concerns, bias in t...
 concerns, bias 

## Semantic + Overlap Chunking

In [6]:
def semantic_overlap_chunking(text, chunk_size=200, overlap=20):
    """Splits text into chunks based on semantic boundaries with optional overlap."""
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        # Check if adding the next sentence exceeds the chunk size
        if len(current_chunk) + len(sentence) + 1 <= chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    # Add overlap
    final_chunk = []
    for i in range(len(chunks)):
        final_chunk.append(chunks[i])
        if i < len(chunks) - 1:
            overlap_text = ' '.join(chunks[i].split()[-overlap//2:]) + ' ' + ' '.join(chunks[i+1].split()[:overlap//2])
            final_chunk.append(overlap_text.strip())
    
    print(f"Created {len(final_chunk)} chunks using **Semantic Overlap Chunking**")
    return final_chunk

# Execute
semantic_chunks = semantic_overlap_chunking(text)
for i, chunk in enumerate(semantic_chunks):
    print(f"[cyan] Semantic Chunk {i+1}: [/cyan] {chunk}..." if len(chunk) > 180 else chunk)
            

Created 9 chunks using **Semantic Overlap Chunking**
The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference.
dramatically since its inception in 1956 at the Dartmouth Conference. Early systems focused on symbolic reasoning and rule-based approaches.
Early systems focused on symbolic reasoning and rule-based approaches.
Early systems focused on symbolic reasoning and rule-based approaches. However, progress slowed during the AI winters in the 1970s
[cyan] Semantic Chunk 5: [/cyan] However, progress slowed during the AI winters in the 1970s and 1980s due to limited computing power and data availability.

A major breakthrough occurred in the 2010s with the rise of deep learning and large-scale datasets....
2010s with the rise of deep learning and large-scale datasets. Companies like OpenAI, Google, and Meta developed large language models
Companies like OpenAI, Google, and Meta developed large 

## Parent-Child Chunking

In [8]:
def parent_child_chunking(text, chunk_size=200, overlap=50):
    """Splits text into parent chunks based on sentence boundaries, then creates child chunks with overlap."""
    sentences = text.split('. ')
    parent_chunks = []
    child_chunks = []
    current_parent = ""
    
    for sentence in sentences:
        if len(current_parent) + len(sentence) + 1 <= chunk_size:
            current_parent += sentence + '. '
        else:
            parent_chunks.append(current_parent.strip())
            current_parent = sentence + '. '
    
    if current_parent:
        parent_chunks.append(current_parent.strip())
    
    # Create child chunks with overlap
    for parent in parent_chunks:
        start = 0
        while start < len(parent):
            end = min(start + chunk_size, len(parent))
            child_chunk = parent[start:end]
            child_chunks.append(child_chunk)
            start += (chunk_size - overlap)   # Sliding window with overlap
    
    print(f"Created {len(parent_chunks)} parent chunks and {len(child_chunks)} child chunks using **Parent-Child Chunking**\n")
    return parent_chunks, child_chunks

parent_chunks, child_chunks = parent_child_chunking(text)
for i, chunk in enumerate(parent_chunks[:4]):
    # print(f"[cyan]Parent Chunk {i+1}:[/cyan] {chunk[:180]}..." if len(chunk) > 180 else chunk)
    print(f"[cyan]Parent Chunk {i+1}:[/cyan] {chunk}...")
print()
for i, chunk in enumerate(child_chunks[:4]):
    # print(f"[cyan]Child Chunk {i+1}:[/cyan] {chunk[:180]}..." if len(chunk) > 180 else chunk)
    print(f"[cyan]Child Chunk {i+1}:[/cyan] {chunk}...")

Created 5 parent chunks and 8 child chunks using **Parent-Child Chunking**

[cyan]Parent Chunk 1:[/cyan] The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference....
[cyan]Parent Chunk 2:[/cyan] Early systems focused on symbolic reasoning and rule-based approaches....
[cyan]Parent Chunk 3:[/cyan] However, progress slowed during the AI winters in the 1970s and 1980s due to limited computing power and data availability.

A major breakthrough occurred in the 2010s with the rise of deep learning and large-scale datasets....
[cyan]Parent Chunk 4:[/cyan] Companies like OpenAI, Google, and Meta developed large language models such as GPT series....

[cyan]Child Chunk 1:[/cyan] The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference....
[cyan]Child Chunk 2:[/cyan] ....
[cyan]Child Chunk 3:[/cyan] Early syste

## Late Chunking

In [9]:
def late_chunking(text, chunk_size=200, overlap=50):
    """Splits text into chunks based on sentence boundaries, optimized for late embedding generation."""
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 <= chunk_size:
            current_chunk += sentence + '. '
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + '. '
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    print(f"Created {len(chunks)} chunks using **Late Chunking**\n")
    return chunks

late_chunks = late_chunking(text)
for i, chunk in enumerate(late_chunks[:4]):
    print(f"[cyan]Late Chunk {i+1}:[/cyan] {chunk[:180]}..." if len(chunk) > 180 else chunk)

Created 5 chunks using **Late Chunking**

The Evolution of Artificial Intelligence. Artificial Intelligence has transformed dramatically since its inception in 1956 at the Dartmouth Conference.
Early systems focused on symbolic reasoning and rule-based approaches.
[cyan]Late Chunk 3:[/cyan] However, progress slowed during the AI winters in the 1970s and 1980s due to limited computing power and data availability.

A major breakthrough occurred in the 2010s with the ris...
Companies like OpenAI, Google, and Meta developed large language models such as GPT series.
